# 技能5 · Day 2 上机：用 LangGraph 编排多Agent营销系统

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库 LangGraph）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 LangGraph 的 `StateGraph` / `Node` / `Edge` 装配一个有状态有向图
2. 用 `add_conditional_edges` 实现"审核不通过回到内容Agent重生成"的条件循环
3. 用 `interrupt` 实现人机协作（HITL）审核

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：LangGraph（`pip install langgraph langchain-anthropic`），营销映射见下。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：
- `langgraph`：状态图框架
- `langchain-anthropic`：Claude LLM 接口（或用 `langchain-openai` 替代）
- 需配置 `ANTHROPIC_API_KEY` 环境变量

In [ ]:
# !pip install langgraph langchain-anthropic langchain-core -q

## 1. LangGraph 背景与营销映射

**LangGraph** 是 LangChain 团队官方的 Agent 编排框架（38k★，MIT），把 Agent 工作流建模为**有状态有向图**：节点是函数（State -> dict），边定义流转，条件边实现循环与分支。

本 Day 的多Agent营销系统工作流：

```
[分析Agent] -> [策略Agent] -> [内容Agent] -> [审核节点]
                                                   |
                                    +------+-------+-------+
                                    |                      |
                                  通过                   不通过
                                    |                      |
                                    v                      v
                               [发布] <----- 回到 [内容Agent]（带审核反馈）
```

**营销映射 × 天道推演**：

| 节点 | 营销职能 | 天道推演对应 |
|------|---------|-------------|
| analysis_agent | 市场分析师 | 局势感知 |
| strategy_agent | 策略总监 | 沙盘模拟 |
| content_agent | 创意文案 | 最优路径推荐 |
| review_node | 合规审核 | 反馈学习 |
| should_approve | 通过/不通过路由 | 概率评估 |

**关键设计**：所有节点共享 `MarketingState`，每个节点只更新自己负责的字段。

In [ ]:
import operator
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage, AIMessage

# LLM 初始化（二选一，取消注释其一；需配置对应 API key）
try:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.7)
    print(f"LLM 已初始化: {llm.model}")
except Exception as e:
    llm = None
    print(f"⚠️ LLM 未初始化（{e}）。图仍可编译，但 stream 执行需配置 ANTHROPIC_API_KEY。")

## TODO 1：定义 MarketingState（全局共享状态）

In [ ]:
# TODO 1：定义 MarketingState（全局共享状态）
# 提示：用 TypedDict 定义，字段如下：
#   brief(str)               -- 用户输入的营销Brief
#   market_analysis(str)     -- 分析Agent输出
#   strategy(str)            -- 策略Agent输出
#   content(str)             -- 内容Agent输出
#   review_feedback(str)     -- 审核反馈
#   revision_count(int)      -- 修改次数（循环退出用）
#   approved(bool)           -- 是否通过审核
#   final_output(str)        -- 最终输出
#   messages(Annotated[list, operator.add])  -- 消息历史（自动累加）
# 要求：完整定义所有9个字段

# ===== 你的代码 =====
class MarketingState(TypedDict):
    brief: str
    pass  # TODO: 补全其余8个字段
# ====================

print("MarketingState 定义完成")

## TODO 2：实现 analysis_agent 和 strategy_agent（两个 LLM 节点）

**节点模式**：`def agent(state: MarketingState) -> dict`
- 读 State 中的字段，拼接 prompt
- 调 `llm.invoke(prompt)`，取 `response.content`
- 返回 dict 更新 State（含 `messages` 用 `AIMessage` 包装）

In [ ]:
# TODO 2：实现 analysis_agent 和 strategy_agent
# 提示：
#   - analysis_agent: 读 state['brief']，prompt 要求输出目标人群/竞品/趋势
#   - strategy_agent: 读 state['market_analysis']，prompt 要求输出核心主张/定位/渠道/信息层级/预算
#   - 返回 {"字段名": response.content, "messages": [AIMessage(content=f"[Agent名]\n{response.content}", name="agent名")]}
# 要求：两个节点都实现完整 prompt 和返回值

# ===== 你的代码 =====
def analysis_agent(state: MarketingState) -> dict:
    """市场分析Agent：分析Brief，提取目标人群、竞品、市场趋势"""
    prompt = None  # TODO: 用 f-string 写 prompt，包含 state['brief']
    response = None  # TODO: llm.invoke(prompt)
    return None  # TODO: 返回 State 更新

def strategy_agent(state: MarketingState) -> dict:
    """策略Agent：基于市场分析制定营销策略"""
    prompt = None  # TODO: 用 f-string 写 prompt，包含 state['market_analysis']
    response = None  # TODO
    return None  # TODO
# ====================

## TODO 3：实现 content_agent（带审核反馈的循环内容生成）

In [ ]:
# TODO 3：实现 content_agent
# 提示：
#   - 检查 state.get("review_feedback") 是否非空：
#       若非空 -> 在 prompt 中加入"上一版内容"和"审核反馈"，要求针对性修改
#       若为空 -> 首次生成
#   - prompt 要求生成3套创意方案（标题/正文/CTA/渠道）
#   - 返回 {"content": response.content, "messages": [AIMessage(..., name="content_agent")]}
# 要求：处理"首次生成"和"根据反馈重生成"两种情况

# ===== 你的代码 =====
def content_agent(state: MarketingState) -> dict:
    """内容Agent：基于策略生成创意内容，若有审核反馈则针对性修改"""
    feedback_section = None  # TODO: 若 state.get("review_feedback") 非空，拼接反馈和上一版 content
    prompt = None  # TODO: 用 f-string 写 prompt，包含 state['strategy'] 和 feedback_section
    response = None  # TODO
    return None  # TODO
# ====================

## 2. 条件路由与循环退出

**条件路由**是 LangGraph 实现循环与分支的关键，由两部分组成：

1. **条件函数**：`def should_approve(state) -> Literal["publish", "revise"]`
   - 读 State 决定下一节点
   - **循环退出条件（必做）**：`revision_count >= 3` 时强制 publish，防止无限循环
2. **条件边注册**：`workflow.add_conditional_edges("review", should_approve, {"publish": "publish", "revise": "content"})`

> ⚠️ 独立教材 3.2.4 实践建议第 3 条：**任何循环都必须有退出条件**，否则 Agent 会陷入无限循环。

## TODO 4：实现 review_node 和 should_approve（审核 + 条件路由）

In [ ]:
# TODO 4：实现 review_node 和 should_approve
# 提示：
#   review_node:
#   - prompt 要求对 state['content'] 打4维分（品牌调性/事实准确性/合规性/创意度，各1-10分）
#   - 全>=7判通过；输出格式含"通过状态：通过"或"通过状态：不通过"
#   - 返回 {"approved": bool, "review_feedback": content,
#           "revision_count": state.get("revision_count",0)+1, "messages": [...]}
#   - 解析 approved：检查 response.content 是否包含"通过状态：通过"
#
#   should_approve(state) -> Literal["publish", "revise"]:
#   - 若 revision_count >= 3 -> "publish"（强制退出循环）
#   - 否则 approved=True -> "publish", approved=False -> "revise"
# 要求：两个函数都实现

# ===== 你的代码 =====
def review_node(state: MarketingState) -> dict:
    """审核节点：检查内容合规性和质量"""
    prompt = None  # TODO
    content = None  # TODO: llm.invoke(prompt).content
    approved = None  # TODO: 解析 content 判断是否通过
    return None  # TODO

def should_approve(state: MarketingState) -> Literal["publish", "revise"]:
    """条件路由：根据审核结果决定发布还是修改"""
    # TODO: 循环退出 + 条件判断
    pass
# ====================

## 3. 发布节点（给定，无需填写）

`publish_node` 是工作流的终点，把所有 Agent 的输出汇总成最终方案。这里直接给出，无需你填写。

In [ ]:
def publish_node(state: MarketingState) -> dict:
    """发布节点：生成最终输出"""
    final_output = f"""
==================================================
        营销方案最终输出
==================================================

【营销Brief】
{state['brief']}

【市场分析】
{state['market_analysis']}

【营销策略】
{state['strategy']}

【创意内容】
{state['content']}

【审核记录】
修改次数：{state.get('revision_count', 0)}
最终审核：{'通过' if state.get('approved') else '超过修改上限，强制发布'}

==================================================
"""
    return {
        "final_output": final_output,
        "messages": [AIMessage(content="[发布节点] 内容已发布", name="publish_node")]
    }

## TODO 5：实现 build_marketing_graph（StateGraph 装配）

In [ ]:
# TODO 5：实现 build_marketing_graph
# 提示：
#   - workflow = StateGraph(MarketingState)
#   - add_node: 5个节点 -- "analysis"(analysis_agent), "strategy"(strategy_agent),
#               "content"(content_agent), "review"(review_node), "publish"(publish_node)
#   - add_edge(普通边): START->analysis, analysis->strategy, strategy->content,
#                       content->review, publish->END
#   - add_conditional_edges("review", should_approve, {"publish": "publish", "revise": "content"})
#   - graph = workflow.compile(checkpointer=MemorySaver())
# 要求：返回编译后的 graph

# ===== 你的代码 =====
def build_marketing_graph():
    """构建营销Agent系统的LangGraph"""
    workflow = None  # TODO: StateGraph(MarketingState)
    # TODO: add_node（5个节点）
    # TODO: add_edge（普通边：START->analysis->strategy->content->review, publish->END）
    # TODO: add_conditional_edges("review", should_approve, {...})
    graph = None  # TODO: workflow.compile(checkpointer=MemorySaver())
    return graph
# ====================

# 测试图是否能编译（不需要 LLM）
try:
    test_graph = build_marketing_graph()
    print("✓ 图编译成功！节点:", list(test_graph.get_graph().nodes.keys()))
except Exception as e:
    print(f"✗ 图编译失败：{e}")

## 4. 运行系统（给定，无需填写）

`run_marketing_agent` 把图跑起来：初始化 State、配置线程ID（检查点用）、流式执行并打印每个节点的输出。这里直接给出。

In [ ]:
def run_marketing_agent(brief: str):
    """运行营销Agent系统"""
    graph = build_marketing_graph()
    initial_state = {
        "brief": brief,
        "market_analysis": "",
        "strategy": "",
        "content": "",
        "review_feedback": "",
        "revision_count": 0,
        "approved": False,
        "final_output": "",
        "messages": [],
    }
    config = {"configurable": {"thread_id": "marketing_session_001"}}
    print("启动营销Agent系统...\n")
    for event in graph.stream(initial_state, config=config, stream_mode="values"):
        if event.get("messages"):
            latest_msg = event["messages"][-1]
            if hasattr(latest_msg, "name") and latest_msg.name:
                print(f"\n{'='*60}")
                print(f"[{latest_msg.name}]")
                print(f"{'='*60}")
                text = latest_msg.content
                print(text[:500] + "..." if len(text) > 500 else text)
    final_state = graph.get_state(config)
    print("\n" + "="*60)
    print("最终输出：")
    print("="*60)
    print(final_state.values.get("final_output", "无输出"))
    return final_state.values

## 5. 人机协作（HITL）：用 interrupt 实现真人工审核

上面的 `review_node` 用 LLM 自动审核。生产环境中，审核应由人工完成。LangGraph 的 `interrupt` 功能可以让图在审核节点**暂停**，State 被检查点持久化，人工给出通过/不通过后图从检查点**恢复**执行。

```python
from langgraph.types import interrupt, Command
# interrupt(value) 暂停图，把 value 抛给调用方；人工 ressume 时传入结果
```

这是 LangGraph 相对其他框架的核心优势，也是生产环境审核合规的关键能力。

## TODO 6（可选）：人工审核 + 运行系统

In [ ]:
# TODO 6（可选）：把 review_node 替换为 interrupt 真人工审核，并运行系统
# 提示：
#   human_review_node:
#   - 用 interrupt({"question": ..., "content": state["content"], "options": [...]}) 暂停图
#   - interrupt 返回值是人工 ressume 时传入的结果（如 "通过" / "不通过-需修改"）
#   - 返回 {"approved": bool, "review_feedback": review_result}
#   - 注意：用 human_review_node 替换 review_node 后，需重新 build_marketing_graph
# 要求：实现 human_review_node + 运行完整系统（若 LLM 已初始化）

# ===== 你的代码 =====
def human_review_node(state: MarketingState) -> dict:
    """真正的人工审核节点：暂停执行，等待人工输入"""
    review_result = None  # TODO: interrupt({"question": ..., "content": ..., "options": [...]})
    approved = None  # TODO: 判断是否通过
    return None  # TODO

# 运行系统
brief = """
品牌：雅净（Yajing）
产品：新款烟酰胺精华液
目标：新品上市推广，3个月内品牌知名度提升20%
预算：50万元
渠道：小红书、抖音、天猫
"""
# TODO: 若 LLM 已初始化，取消下面注释运行
# result = run_marketing_agent(brief)
# ====================

## 6. 反思与前沿

### 反思问题
1. 你的条件循环退出条件（`revision_count >= 3`）设成多少合理？为什么？如果取消退出条件会发生什么？
2. `MarketingState` 为什么要用 `Annotated[list, operator.add]` 处理 `messages` 字段？（提示：默认行为是覆盖，用 `operator.add` 改为累加，让每个节点的消息都保留）
3. `MemorySaver` 和 `SqliteSaver` 的区别是什么？生产环境为什么必须用后者？（提示：内存 vs 持久化，服务重启后状态是否丢失）
4. 如果要在分析Agent之前加一个"用户画像Agent"节点，需要改哪些地方？（提示：加节点函数 + add_node + 改 edge，体现 LangGraph 的可扩展性）

### 2026 前沿：多Agent仿真 × 天道推演
LangGraph 多Agent编排与项目 CLAUDE.md 的「天道推演系统」高度同构--天道推演在意识中构建多路径沙盘，LangGraph 在代码中构建多Agent状态图。

**升级路径**：可把天道推演从"思维框架"升级为**可计算多Agent沙盘**--用 LangGraph 模拟多个利益相关方 Agent（品牌方/渠道方/消费者/竞品方）博弈，每个 Agent 是一个节点，条件边模拟博弈分支，推演不同决策路径下的结果分布。

| 天道推演（思维框架） | LangGraph（可计算实现） |
|--------------------|-----------------------|
| 在意识中构建多路径沙盘 | 在代码中构建多Agent状态图 |
| 模拟不同决策路径下的未来走向 | 条件边展开多分支执行 |
| 选择最优路径或预判风险 | 评估各分支输出择优 |
| 记录假设、追踪偏差、更新模型 | Checkpointing 持久化 + 反馈学习节点 |

参考：项目 CLAUDE.md「天道推演系统」｜ LangGraph 多Agent协作 https://github.com/langchain-ai/langgraph ｜ LangChain Academy 官方课程 https://github.com/langchain-ai/langchain-academy

> 🔗 深入阅读见 `reading.md` 的"多Agent仿真 × 天道推演"条目。